# Per-embryo expression profiles (Fig. 2b, d)

Pools the UMIs of each embryo x cell type into one profile, the input of the divergence scores in `Fig2.ipynb` (`find_best_matches`, `scripts/rna_projection_func.r`).

- **2b**: divergence scores, epiblast, query vs best-matched atlas embryos
- **2d**: divergence scores, early nascent mesoderm, query vs best-matched atlas embryos

Inputs:
- `./data/MC_models/mars_<dataset>_ff_cells.h5ad` and `..._metacells_projected.h5ad`
- WT gastrulation atlas (spatwt)

Outputs (under `git_test/`, through `out_path()`):
- `./data/gc_per/gc_per_embryo_<dataset>.h5ad` (one per query dataset)
- `./data/gc_per/gc_per_embryo_atlas.h5ad`

In [ ]:
# Paths come from ../paths.R (the R notebooks source it directly).
import os, re
repo_dir = os.path.abspath("..")
_paths = open(os.path.join(repo_dir, "paths.R")).read()
def r_path(name):
    return re.search(r'^%s <- "([^"]+)"' % name, _paths, re.M).group(1)
os.chdir(r_path("embdnmt_dir"))

# As out_path() in paths.R: writes go to git_test/ in the repo, keeping their path relative to embdnmt_dir.
def out_path(p):
    f = os.path.join(repo_dir, "git_test", os.path.relpath(os.path.abspath(p), os.path.abspath(r_path("embdnmt_dir"))))
    os.makedirs(os.path.dirname(f), exist_ok=True)
    return f

In [5]:
import anndata as ad
import metacells as mc
import numpy as np
import pandas as pd
import scipy.sparse as sp  # type: ignore
import scipy.stats as ss  # type: ignore
from anndata import AnnData  # type: ignore
import metacells.utilities as ut

import re

## Query embryos

- Cell type: the metacell's `projected_type` on the WT atlas; cells in no metacell are `Outliers`.
- Pools with `mc.pl.collect_metacells`.
- Groups with ≤ 100,000 UMIs get `embryo` and `cell_type` set to `None`, so `Fig2.ipynb` ignores them.
- Datasets: TKO and DKO 4N mutants, KO and host fractions of both chimeras.
- Output name drops the `_ff` suffix.

In [6]:
def gc_per_emb(nm, cells, mcells):

    mc_to_type = mcells.obs["projected_type"].to_dict()

    cells_type_list = []
    for i in cells.obs.metacell_name:
        if i != "Outliers":
            cells_type_list.append(mc_to_type[i])
        else:
            cells_type_list.append("Outliers")

    mc.ut.set_o_data(cells, "projected_type", cells_type_list)
    mc.ut.set_o_data(cells, "label_string" , 
                     cells.obs.embryo.astype(str) + "..." + cells.obs.projected_type.astype(str))

    value_counts = cells.obs.label_string.value_counts()

    df = pd.DataFrame(cells.X)

    df['label_string'] = cells.obs.label_string.tolist()
    umis_per_id = df.groupby('label_string').sum()

    umis_per_id_df = pd.DataFrame(umis_per_id)
    umis_per_id_df.columns = cells.var_names

    enough_umis = umis_per_id.sum(axis=1) > 1e+5
    underrepresented_mask = ~enough_umis[cells.obs.label_string]

    label_string_to_int = {}
    for i ,j in enumerate(cells.obs.label_string.unique()):
        label_string_to_int[j] = i

    mc.ut.set_o_data(cells, "label_int" ,[label_string_to_int[i] for i in cells.obs.label_string])

    gc_per_embryo = mc.pl.collect_metacells(cells, groups="label_int", random_seed=12345, name="label_string", prefix="")

    under_represented_ids = [key for key, value in underrepresented_mask.to_dict().items() if value == True]

    obs_index = [str(value) for value in label_string_to_int.values()]
    gc_per_embryo.obs = pd.DataFrame(list(label_string_to_int.keys()), index=obs_index, columns=["id"])
    gc_per_embryo.obs[['embryo', 'cell_type']] = [x.split("...") if x not in under_represented_ids else [None, None] for x in gc_per_embryo.obs['id'].tolist()]

    gc_per_embryo.var = pd.DataFrame(index=cells.var_names)
    mc.ut.get_vo_frame(gc_per_embryo)

    gc_per_embryo.write(out_path("./data/gc_per/gc_per_embryo_" + re.sub(r"_ff$", "", nm) + ".h5ad"))

In [7]:
for name in ["DNMT_TKO_4N_ff", "DNMT_DKO_4N_ff", "DNMT_TKO_KO_ff", "DNMT_TKO_host_ff", "DNMT_DKO_KO_ff", "DNMT_DKO_host_ff"]:
    cells = ad.read_h5ad("./data/MC_models/mars_" + name + "_cells.h5ad")
    mcells = ad.read_h5ad("./data/MC_models/mars_" + name + "_metacells_projected.h5ad")
    
    gc_per_emb(nm = name, cells = cells, mcells = mcells)

## WT atlas embryos

Same pooling with the atlas' own `cell_type` (cells outside the model are `non-model`), written to `./data/gc_per/gc_per_embryo_atlas.h5ad`.

- Caveat: never run in the stored notebook.

In [ ]:
ref_atlas_path = r_path("spatwt_dir") + "/"

mc_wt = ad.read_h5ad(ref_atlas_path + "spatwt.metacells.h5ad")
cells_wt = ad.read_h5ad(ref_atlas_path + "spatwt.cells.h5ad")

In [ ]:
mc_to_type = mc_wt.obs["cell_type"].to_dict()
mc_set = set(mc_to_type)

cells_type_list = []
for i in cells_wt.obs.metacell_name:
    if i != "Outliers" and i in mc_set:
        cells_type_list.append(mc_to_type[i])
    else:
        cells_type_list.append("non-model")

mc.ut.set_o_data(cells_wt, "cell_type", cells_type_list)
mc.ut.set_o_data(cells_wt, "label_string" , 
                 cells_wt.obs.embryo.astype(str) + "..." + cells_wt.obs.cell_type.astype(str))

value_counts = cells_wt.obs.label_string.value_counts()

In [ ]:
df = pd.DataFrame(cells_wt.X)

df['label_string'] = cells_wt.obs.label_string.tolist()
umis_per_id = df.groupby('label_string').sum()

In [ ]:
umis_per_id_df = pd.DataFrame(umis_per_id)
umis_per_id_df.columns = cells_wt.var_names

In [ ]:
enough_umis = umis_per_id.sum(axis=1) > 1e+5
underrepresented_mask = ~enough_umis[cells_wt.obs.label_string]

In [ ]:
label_string_to_int = {}
for i ,j in enumerate(cells_wt.obs.label_string.unique()):
    label_string_to_int[j] = i

mc.ut.set_o_data(cells_wt, "label_int" ,[label_string_to_int[i] for i in cells_wt.obs.label_string])

gc_per_embryo = mc.pl.collect_metacells(cells_wt, groups="label_int", random_seed=12345, name="label_string", prefix="")

In [ ]:
under_represented_ids = [key for key, value in underrepresented_mask.to_dict().items() if value == True]

In [ ]:
obs_index = [str(value) for value in label_string_to_int.values()]
gc_per_embryo.obs = pd.DataFrame(list(label_string_to_int.keys()), index=obs_index, columns=["id"])
gc_per_embryo.obs[['embryo', 'cell_type']] = [x.split("...") if x not in under_represented_ids else [None, None] for x in gc_per_embryo.obs['id'].tolist()]

gc_per_embryo.var = pd.DataFrame(index=cells_wt.var_names)
mc.ut.get_vo_frame(gc_per_embryo)

gc_per_embryo.write(out_path('./data/gc_per/gc_per_embryo_atlas.h5ad'))